# Step 1 Phase A-8b：走査ベンチマーク（staged・隔離 subprocess・Colab 実環境）**v1.1.8**
2026-09-10。**v1.1.7 監査（science core APPROVED・formal archive の 2 BLOCKER）**：(1) amendment 報告書の **期待 SHA256 を hard gate**（存在検査のみだった）；(2) **timed audit vectors を正式成果物に保存**（`a8b_timed_audit_vectors.npz`＋`a8b_timed_audit_index.json`：全 timed config の argmin/selection score/T₁/T₂[0:AUDIT_N]；reopen して in-memory と一致することを gate；provenance `outputs` に SHA）——これで `G_audit_matches_timed`・`G_chunk_invariance`・`G_float64_raw_argmin_exact`・exact-antipode flip を第三者が成果物から再計算できる。
推奨 hardening：antipode map の **期待 SHA**（`11efe112…`，独立構成値と一致）＋幾何 gate（max|vec[A]+vec| < 1e-12）／親 FAILED smoke の **raw artifacts**（audit NPZ・evidence JSONL）も親 provenance の `outputs` SHA に照合（`G_parent_failed_smoke_artifacts`）／cheap gate（amendment・親 artifact・antipode・Event B 閾値・述語 self-test）を **セル 1 の preflight** に移動（fail-fast）／Step 0 行の一意性（`len(rows)==1`）。
provenance 表現：`float32_flip_count` → `float32_flip_occurrences_across_chunk_comparisons`（config 間で同一標本を重複計上する量）＋ route 別 unique flipped sample 数；rules statement を「antipodal pair が **exact implemented tie** の場合」の条件形に。amendment 報告書 `A8b_amendment_v1.1.7.md` は監査済み bytes（SHA `950bea6c…`）のまま固定し，v1.1.8 はその amendment の実装＋archive hardening。数値カーネル・staged 設計・閾値・winner 規則は不変。
v1.1.7：2026-09-10。**v1.1.6 監査（tie-equivalence gate の訂正・amended analysis）**：v1.1.6 の margin-aware gate は plane equivalence を hard gate しておらず（raw margin は対蹠重複で常に 0 のため，対蹠でない遠い軸への flip も通り得た），
失敗分析書の合成試験は notebook の述語に対するものではなく，float64 側まで不要に緩和し，chunk gate の reference axis（pilot）と margin source（winner audit）が食い違っていた。v1.1.7：
(1) **共通 helper `selected_output_equiv`**——float64：raw argmin exact／float32：same または **exact antipode かつ raw margin < 選択許容**；selection score・float64 T₁/T₂・**Event B indicator** の一致——を audit 照合・chunk 不変性・direct vs feature・f32 vs f64・GPU vs CPU の**全比較で共有**。
(2) canonical reference を untimed audit child に統一（reference axis・runner-up margin・selected outputs が同一計算から来る）。(3) required gate 追加：`G_float64_raw_argmin_exact`・`G_antipode_map_involution`（antipode map SHA を provenance へ）・
`G_plane_equivalence_gate_selftest`（6 合成ケース）・`G_eventB_thresholds_bound`（Step 0 official CSV の凍結閾値）・`G_parent_failed_smoke_binding`（親 FAILED smoke の path/SHA/false gates・amendment 報告書 SHA・旧/新 gate 意味）。
(4) provenance の再現性記述は固定文字列ではなく実測値から生成（`selection_reproducibility.measured`），margin は `raw_oriented_axis_margin` と定義を明記，`RULES_RECOMMENDATION.axis_reproducibility` を追加。
数値カーネル・staged 設計・selection float32／evaluation float64・memory／checkpoint 設計・閾値（1e-4／1e-9／1e-6）・winner 規則は不変。v1.1.5 smoke は FAILED のまま保存し，本版は新規 commit → 新規 smoke → 新規 official。
v1.1.6（撤回）：v1.1.5 の Colab smoke（2 threads）で `G_chunk_invariance`／`G_audit_matches_timed` が **float32 selection のみ**で失敗（float64 は全 chunk で bit 同一）。内訳：selection score 差 ≤7.4e-7（float32 分解能）・**T₁/T₂ 差 0.0**・cross 不一致の margin はすべて 0.0（厳密同点）
——対蹠軸（B 行同一・反射面同一）の**厳密同点**で，マルチスレッド GEMM のブロッキングが chunk により変わり 1e-7 の丸め差で勝者が入れ替わる現象。バグではなく，A5/A8a で既知の対蹠同点と float32 の相互作用。
v1.1.6：(1) chunk 不変性と audit 照合を **margin-aware** に（軸差は margin < 選択許容 かつ T₁/T₂ 一致 (TOL_EVAL) の場合のみ許容；flip 率と flip 先が対蹠か否かを記録），(2) audit child は timed run と同じ chunk ブロッキングで処理（先頭 AUDIT_N を保持），
(3) provenance/RULES に `float32_selection_reproducibility` を記録：production の chunk・thread は登録定数，plane-equivalent 同点の argmin は chunk/thread 構成間で再現しない，axis 出力は plane-folded で扱う，T₁/T₂/Event B は不変。
v1.1.5：v1.1.4 監査（#6）：最終 assert を mode 別に（v1.1.4 では `assert BENCHMARK_VALID` が残り smoke が必ず失敗した——私のテスト出力の誤読）／GPU 採用に CPU S2 f32-vs-f64 PASS・`hashes_consistent`・timing finite を要求（`gpu_eligible()`）／
chunk 不変性 coverage は **distinct (chunk, axis_block) ≥ 2** を要求し，T₁/T₂ は TOL_EVAL=1e-6／BENCH_ENV に RAM/swap total（cgroup limit があれば）を追加／child で `threadpool_limits` を併用／最終出力も atomic write／`kernel_s_median` alias を除去し表示を production に。
v1.1.4：v1.1.3 監査（#5）：smoke/official の status 分離（`SMOKE_PASS` vs `BENCHMARK_VALID`）／GPU probe は常に uncached で実行し，その戻り値（GPU model・VRAM・torch・CUDA）と GPU child SHA から **GPU_BINDING** を確定して pilot 以降の checkpoint を bind／
GPU child の host float64 評価も BLAS thread を固定し threadpool を返す（採用条件に含める）／GPU child も B-stack SHA を再検査／fallback は (chunk, axis_block) で dedup（代表 = 最大 N → finalist/winner 優先）／
A8b は両 mode とも **A8a official artifact** を読む／T₁/T₂ の cross 許容 1e-6 を selection 許容 1e-4 と分離／peak memory の tie は raw 値／timing gate を全成分に。
v1.1.3：v1.1.2 監査（#4）の BLOCKER：**selection dtype と evaluation dtype を分離**——A5 凍結仕様（selection float32 primary／evaluation float64）どおり，全 route で sample は float64 で生成し，
selection のみ selection dtype に cast，T₁/T₂ は**選択軸で float64 x4 と float64 B-stack** で評価（GPU も axis を D2H 後に host float64 で評価）。時間は draw/cast/scan/eval64/production/end-to-end に分離し，選択は production_s で行う。
A8a の 2 flag（event/axis）を両方受け取り provenance・ROUTE_DECISION に保存／audit child と timed child の 4 出力照合／audit vectors を NPZ に保存／chunk 不変性の coverage gate／marker は max 更新／GPU も checkpoint。
v1.1.2：v1.1.1 監査（#3）の 3 BLOCKER：**監査用 runner-up 計算を timed kernel から完全に分離**（route/dtype ごとに 1 回の untimed audit child）／ℓ2–4 pilot N を 4×10⁴（≥2×max chunk）に／
checkpoint binding に benchmark 環境（Python・NumPy・BLAS・CPU model・platform・affinity・A8a provenance/manifest SHA・float32 適格性）を含める。GPU：pilot の safe tie-break・optional 1e6 の線形性＋予算規則・VRAM 65% 安全閾値。
route 間 5% tie は低 peak memory → 登録 priority（direct）で決定。atomic checkpoint（fsync）・optional 1e6 も checkpoint。
v1.1.1：v1.1 監査（#2）：登録した **5% safe tie-break を実装**（finalist→winner・最終 pick）／route ごとの dtype 適格性（feature231 も f32-vs-f64 電池に）／GPU 採用は A8a `float32_eligible_for_rules` ∧ 採用 config 自身の cross-check PASS／
thread 数の**登録値一致** gate／peak は sampled・current・`ru_maxrss` の最大値で判定＋明示 marker／child が runner-up gap（真の margin）を返す／optional 1e6 成功時は decision を実測で更新／fallback は (chunk, axis_block, N)／per-config checkpoint・resume／full hash・audit 要約を JSONL に保存。
v1.0 監査（2026-09-09）の 10 BLOCKER に対応。**登録済み adaptive staged design（結果を見る前に固定）**：

- **Stage 1 pilot**：全 chunk/axis-block × N（ℓ2–4：4×10⁴・S2：10⁴）・warm-up 2 chunk・repeat 1。finalist は raw top-K＋safe-choice。
- **Stage 2 finalist**：route/dtype ごとに pilot 上位 2（S2 は上位 3）× N=10⁵・repeat 3。
- **Stage 3 winner**：route/dtype ごとの上位 1 × N（ℓ2–4：10⁶／S2：2×10⁵）・repeat 3。S2 N=10⁶ は「N≤2×10⁵ の線形性（per-sample 比 <1.2）＋予算 40 分」を満たす全体 winner のみ実測。
- kernel：S⁺ 全軸 → argmin → **S⁻ は選択軸のみ**。子プロセスは argmin・**selection score**・T₁・T₂ を返し，chunk/axis-block 不変性は 4 量で gate。
- 各 attempt を `ok/oom/timeout/killed/numerical_fail/other_error` に分類して必ず保存（OOM は ineligible であって benchmark 失敗ではない）。exact registered inventory（canonical key・重複なし・optional N=10⁶ は別集合）。
- F16 は **明示的 resident copy**（`np.array(copy=True)`・`shares_memory` を assert）。production 実装も resident と登録。
- BLAS thread 数を子プロセスに固定（affinity 数・registered），子が `threadpool_info()` を返し一定性を gate。
- メモリ eligibility：`peak_increment ≤ 0.65 × available_before_setup` かつ swap 増加 0。peak は 5 ms sampling＋明示 marker の最大（`ru_maxrss` は fork で親の high-water mark を継承するため delta 診断のみ）。tie-break は (chunk, axis_block) 昇順（安全側）。
- cross-route：ℓ2–4 direct vs feature（f64・f32）で argmin・selected S⁺・S⁻ を margin-aware に比較。GPU strict vs CPU f32 も同様。
- **ROUTE_DECISION を一意に確定**（ℓ2–4：route/dtype/chunk・S2：CPU or GPU・resident・fallback tiers）。dtype は A8a の `float32_eligible_for_rules` ∧ A8b cross-route 一致のときのみ float32。
- source/input chain：notebook live vs origin/main・repo pin・child script SHA・A8a provenance/manifest/F16 SHA（child が timing 前に再照合）。

In [ ]:
# ---- 1. 環境・source identity・A8a artifact chain ----
import os, sys, json, hashlib, subprocess, time, datetime, platform, textwrap, glob, importlib
from importlib.metadata import version as pkg_version, PackageNotFoundError
A8_MODE = globals().get('A8_MODE', os.environ.get('A8_MODE', 'official')); assert A8_MODE in ('smoke', 'official')
IN_COLAB = os.path.isdir('/content'); WORK = '/content' if IN_COLAB else os.environ.get('A8_WORK', os.path.join(os.getcwd(), 'a8_work'))
BASE = '/content/drive/MyDrive/mirror_topology' if IN_COLAB else os.environ.get('A8_BASE', os.path.join(WORK, 'base'))
if IN_COLAB:
    if not os.path.isdir('/content/drive/MyDrive'):
        from google.colab import drive; drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'psutil', 'threadpoolctl', 'healpy==1.20.0'], check=True)
A8A = os.path.join(BASE, 'runs_step1_phaseA', os.environ.get('A8A_DIR', 'a8a_v1.1.2_official'))   # both modes preflight the OFFICIAL A8a artifact
OUT = os.path.join(BASE, 'runs_step1_phaseA', f'a8b_v1.1.8_{A8_MODE}'); LOCAL = os.path.join(WORK, 'a8b_local')
for d in (OUT, LOCAL): os.makedirs(d, exist_ok=True)
import numpy as np, pandas as pd, psutil
from threadpoolctl import threadpool_info
GATES = {}; DIAG = {}; GIT_LOG = []
def sha256_file(p, block=8 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(block), b''): h.update(b)
    return h.hexdigest()
def sha256_array(a, block=8 << 20):
    a = np.ascontiguousarray(a); mv = memoryview(a).cast('B'); h = hashlib.sha256()
    for i in range(0, len(mv), block): h.update(mv[i:i + block])
    return h.hexdigest()
def run(c, **kw):
    r = subprocess.run(c, capture_output=True, text=True, check=True, **kw); GIT_LOG.append(dict(cmd=c, stderr=r.stderr.strip()[:200])); return r.stdout.strip()
_env = dict(os.environ, GIT_TERMINAL_PROMPT='0'); MT = os.path.join(WORK, 'mt_a8'); MT_COMMIT = '5ad38e12c4d769f3ddbfb95b2cff18d959bfc0e7'
if not os.path.isdir(os.path.join(MT, '.git')): run(['git', 'clone', 'https://github.com/tsujikeita/mirror-topology.git', MT], env=_env)
run(['git', '-C', MT, 'fetch', '-q', 'origin', MT_COMMIT], env=_env); run(['git', '-C', MT, 'reset', '-q', '--hard', MT_COMMIT]); run(['git', '-C', MT, 'clean', '-fdx', '-q'])
GATES['G_mt_commit'] = (run(['git', '-C', MT, 'rev-parse', 'HEAD']) == MT_COMMIT); GATES['G_mt_origin'] = (run(['git', '-C', MT, 'remote', 'get-url', 'origin']).rstrip('/').removesuffix('.git') == 'https://github.com/tsujikeita/mirror-topology')
GATES['G_mt_clean'] = (run(['git', '-C', MT, 'status', '--porcelain']) == ''); GATES['G_t2b2_run_sha'] = (sha256_file(os.path.join(MT, 't2b2_run.py')) == '03c80f2136a8ff7ffb1077749895811ef95dd9d779c7996891d89e75545ff8db')
assert all(GATES.values()), GATES
for name in list(sys.modules):
    if name in {'t2b2_run', 't2b2_bridge', 't1_engine'}: del sys.modules[name]
importlib.invalidate_caches(); sys.path.insert(0, MT); import t2b2_run as tr
NB_BASENAME = 'MirrorTopology_Step1_A8b_scan_benchmark_v1.1.8.ipynb'; run(['git', '-C', MT, 'fetch', '-q', 'origin', 'main'], env=_env); MAIN_HEAD = run(['git', '-C', MT, 'rev-parse', 'origin/main'])
try: NB_HEAD_SHA = tr.source_only_sha(subprocess.run(['git', '-C', MT, 'show', f'origin/main:{NB_BASENAME}'], capture_output=True, check=True).stdout)
except subprocess.CalledProcessError: NB_HEAD_SHA = None
NB_LIVE_SHA = tr.live_notebook_source_sha(); DIAG['notebook'] = dict(live=NB_LIVE_SHA, origin_main_head=MAIN_HEAD, head_copy=NB_HEAD_SHA)
if A8_MODE == 'official': GATES['G_notebook_live_source'] = bool(isinstance(NB_LIVE_SHA, str) and NB_HEAD_SHA is not None and NB_LIVE_SHA == NB_HEAD_SHA); assert GATES['G_notebook_live_source'], DIAG['notebook']
else: GATES['G_notebook_head_available'] = (NB_HEAD_SHA is not None)
# ---- A8a artifact chain ----
a8a_prov_path = os.path.join(A8A, 'a8a_provenance.json'); a8a = json.load(open(a8a_prov_path)); A8A_PROV_SHA = sha256_file(a8a_prov_path)
GATES['G_a8a_status'] = (a8a['status'] == 'FEATURESTACK_VALID' and a8a['gate_inventory_exact'] is True and all(a8a['gates'][k] is True for k in a8a['required_gates']))
man_path = os.path.join(A8A, a8a['manifest_file']); GATES['G_a8a_manifest_sha'] = (sha256_file(man_path) == a8a['manifest_file_sha256']); man = json.load(open(man_path))
GATES['G_a8a_manifest_embedded_equal'] = (man == json.loads(json.dumps(a8a['manifest'])))
F64 = os.path.join(A8A, 's1_Bplus_featurestack_l2_16_N16_common_v1_float64.npy'); F32 = os.path.join(A8A, 's1_Bplus_featurestack_l2_16_N16_common_v1_float32.npy')
GATES['G_F16_file_sha'] = (sha256_file(F64) == man['sha256']['F16_float64_file'] and sha256_file(F32) == man['sha256']['F16_float32_file'])
BST = os.path.join(MT, 'results', 'step1_phaseA', 'A5_freeze', 's1_Bstack_l2_4_N16_common_v1.npz'); GATES['G_a5_bstack_file_sha'] = (sha256_file(BST) == man['inputs']['bstack_file'])
GATES['G_colab_runtime'] = IN_COLAB if A8_MODE == 'official' else True
# ---- amendment provenance (v1.1.7): this notebook amends the tie-equivalence gate AFTER the v1.1.5 smoke FAILED. The change is bound to its evidence:
#      the parent FAILED smoke provenance (path, SHA, false gates) and the amendment report (SHA), plus the old/new gate semantics. Transparent, not silent.
PARENT_SMOKE = os.path.join(BASE, 'runs_step1_phaseA', 'a8b_v1.1.5_smoke', 'a8b_provenance.json'); AMEND_REPORT = os.path.join(BASE, 'runs_step1_phaseA', 'A8b_amendment_v1.1.7.md')
AMEND = dict(parent_failed_smoke_provenance_path=PARENT_SMOKE, parent_failed_smoke_provenance_sha256_expected='ad92387f5bd9cdcd7e6a393688704503660539d1abdb44d7962be4398a418a40',
             parent_notebook_expected='Step1 Phase A-8b scan benchmark v1.1.5 [smoke]', parent_false_gates_expected=['G_audit_matches_timed', 'G_chunk_invariance'], amendment_report_path=AMEND_REPORT,
             old_gate_semantics='chunk/audit invariance required raw HEALPix argmin identity across chunk configurations (all selection dtypes)',
             new_gate_semantics='selected-output equivalence: float64 raw argmin exact; float32 axis may differ only to the exact antipodal partner (same unoriented plane) at raw margin < selection tol; '
                                'selection score (sel tol), float64 T1/T2 (TOL_EVAL) and Event B indicator must agree; one shared predicate for every comparison',
             amendment_report_sha256_expected='950bea6c479c9a786158db530f0242cc659e507b25bbdbaad8f67e5839d7c68b', amended_on='2026-09-10', amended_by='v1.1.7 after ChatGPT audit of the v1.1.6 gate (2026-09-10)',
             implemented_in='v1.1.8 = v1.1.7 amendment + archival hardening after ChatGPT audit of v1.1.7 (report expected SHA, parent raw artifacts, timed audit vectors, antipode expected SHA, preflight)')
try:
    pp = json.load(open(PARENT_SMOKE)); AMEND['parent_failed_smoke_provenance_sha256'] = sha256_file(PARENT_SMOKE); AMEND['parent_status'] = pp['status']; AMEND['parent_notebook'] = pp['notebook']
    AMEND['parent_false_gates'] = [k for k, v in pp['gates'].items() if not v]; AMEND['amendment_report_sha256'] = sha256_file(AMEND_REPORT) if os.path.exists(AMEND_REPORT) else None
    GATES['G_parent_failed_smoke_binding'] = (AMEND['parent_failed_smoke_provenance_sha256'] == AMEND['parent_failed_smoke_provenance_sha256_expected'] and pp['status'] == 'FAILED' and pp['notebook'] == AMEND['parent_notebook_expected']
                                             and sorted(AMEND['parent_false_gates']) == sorted(AMEND['parent_false_gates_expected']) and AMEND['amendment_report_sha256'] == AMEND['amendment_report_sha256_expected'])
    # parent raw artifacts (the evidence behind the amendment) must still exist on Drive with the bytes recorded in the parent provenance
    _pdir = os.path.dirname(PARENT_SMOKE); AMEND['parent_raw_artifacts'] = {f: (sha256_file(os.path.join(_pdir, f)) if os.path.exists(os.path.join(_pdir, f)) else None) for f in ('a8b_audit_vectors.npz', 'a8b_attempts_evidence.jsonl')}
    GATES['G_parent_failed_smoke_artifacts'] = (AMEND['parent_raw_artifacts']['a8b_audit_vectors.npz'] == pp['outputs']['audit_vectors_npz_sha256'] and AMEND['parent_raw_artifacts']['a8b_attempts_evidence.jsonl'] == pp['outputs']['evidence_jsonl_sha256'])
except Exception as e: AMEND['error'] = f'{type(e).__name__}: {e}'; GATES['G_parent_failed_smoke_binding'] = False; GATES['G_parent_failed_smoke_artifacts'] = False
F32_EVENT = bool(a8a['flags']['float32_eligible_for_event_outputs']); F32_AXIS = bool(a8a['flags']['float32_eligible_for_axis_outputs'])
FLOAT32_ELIGIBLE = F32_EVENT and F32_AXIS      # unified production path: require BOTH (rules: event-only paths may use the event flag alone)
assert all(GATES.values()), GATES
# local copies (verified) + array SHA verification of the feature stacks
EXPECTED = dict(F64_file=man['sha256']['F16_float64_file'], F32_file=man['sha256']['F16_float32_file'], F64_array=man['sha256']['F16_float64_array'], F32_array=man['sha256']['F16_float32_array'], bstack_file=man['inputs']['bstack_file'])
for src, key in ((F64, 'F64_file'), (F32, 'F32_file'), (BST, 'bstack_file')):
    dst = os.path.join(LOCAL, os.path.basename(src))
    if not os.path.exists(dst) or sha256_file(dst) != EXPECTED[key]: subprocess.run(['cp', src, dst], check=True)
    assert sha256_file(dst) == EXPECTED[key], dst
GATES['G_F16_array_sha_local'] = (sha256_array(np.load(os.path.join(LOCAL, os.path.basename(F64)), mmap_mode='r')) == EXPECTED['F64_array'] and sha256_array(np.load(os.path.join(LOCAL, os.path.basename(F32)), mmap_mode='r')) == EXPECTED['F32_array'])
assert GATES['G_F16_array_sha_local']
try: cpu_model = [l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')][0]
except Exception: cpu_model = platform.processor()
THREADS = len(os.sched_getaffinity(0))                                              # registered: child BLAS threads = affinity count
ENV = dict(python=sys.version.split()[0], numpy=np.__version__, platform=platform.platform(), cpu_model=cpu_model, cpu_logical=os.cpu_count(), cpu_affinity=THREADS, threads_registered=THREADS,
           ram_GB=round(psutil.virtual_memory().total / 1e9, 2), swap_GB=round(psutil.swap_memory().total / 1e9, 2), threadpools_parent=[{k: v for k, v in t.items() if k in ('internal_api', 'num_threads', 'version')} for t in threadpool_info()])
try: ENV['gpu'] = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True, check=True).stdout.strip()
except Exception: ENV['gpu'] = 'none'
print(GATES); print('float32 eligible (A8a): event', F32_EVENT, '/ axis', F32_AXIS, '-> selection float32 allowed:', FLOAT32_ELIGIBLE); print(json.dumps(ENV, indent=1)[:700])
# ---- PREFLIGHT (v1.1.8: fail-fast, before any benchmark): selected-output equivalence — ONE predicate shared by every comparison (audit vs timed, chunk/axis-block invariance, direct vs feature, f32 vs f64, GPU vs CPU) ----
import healpy as hp
_vec = np.array(hp.pix2vec(16, np.arange(3072))).T; ANTIPODE = hp.vec2pix(16, -_vec[:, 0], -_vec[:, 1], -_vec[:, 2])
GATES['G_antipode_map_involution'] = bool(np.array_equal(ANTIPODE[ANTIPODE], np.arange(3072)) and not np.any(ANTIPODE == np.arange(3072))); DIAG['antipode_map_sha256'] = sha256_array(ANTIPODE.astype(np.int32))
ANTIPODE_SHA_EXPECTED = '11efe112b8f388b851b5218fa1287aafa3e32ce359f98cccb7c6649d79cc599a'   # HEALPix RING Nside=16 antipode map (int32); independently constructed by the auditor (identical)
GATES['G_antipode_map_expected_sha'] = bool(DIAG['antipode_map_sha256'] == ANTIPODE_SHA_EXPECTED and float(np.abs(_vec[ANTIPODE] + _vec).max()) < 1e-12); DIAG['antipode_geometric_max_residual'] = float(np.abs(_vec[ANTIPODE] + _vec).max())
# Event B thresholds: frozen observed values (Step 0 official, PR3_Commander) read from the pinned repo and bound to the A8a constants
_s0 = pd.read_csv(os.path.join(MT, 'results', 'step0_v0.7', 'step0_official_v0_7.csv')); _rows0 = _s0[_s0['map'] == 'PR3_Commander']; _row = _rows0.iloc[0]
T1o, T2o = 39.67178834527284, 259.3375006282747
GATES['G_eventB_thresholds_bound'] = bool(len(_rows0) == 1 and float(_row['Splus']) == T1o and float(_row['Sminus']) == T2o); DIAG['eventB_thresholds'] = dict(T1_obs=T1o, T2_obs=T2o, source='results/step0_v0.7/step0_official_v0_7.csv @ ' + MT_COMMIT, map='PR3_Commander', indicator='T1 <= T1_obs and T2 <= T2_obs')
TOL = dict(float64=1e-9, float32=1e-4); TOL_EVAL = 1e-6
def au(r): return dict(a=r['audit_argmin_sha'], sel=np.array(r['audit_sel']), t1=np.array(r['audit_T1']), t2=np.array(r['audit_T2']), am=np.array(r['audit_argmin']), mg=np.array(r.get('audit_margin', np.full(len(r['audit_sel']), np.inf))))
def rel(x, y): return float(np.max(np.abs(x - y) / np.maximum(np.abs(y), 1e-300)))
def selected_output_equiv(ref, alt, dtype, sel_tol=None, eval_tol=TOL_EVAL):
    # float64: raw argmin exact. float32: an axis may differ ONLY to the exact antipodal partner (same unoriented mirror plane [n]={n,-n}) where the reference raw oriented-axis margin
    # is below the selection tolerance (exact/near tie). Always: selection score (sel_tol), float64 T1/T2 (eval_tol) and the Event B indicator must agree. ref carries the margins.
    sel_tol = TOL[dtype] if sel_tol is None else sel_tol
    same = (alt['am'] == ref['am']); anti = (alt['am'] == ANTIPODE[ref['am']])
    axis_ok = same if dtype == 'float64' else (same | (anti & (ref['mg'] < sel_tol)))
    eb_ref = (ref['t1'] <= T1o) & (ref['t2'] <= T2o); eb_alt = (alt['t1'] <= T1o) & (alt['t2'] <= T2o)
    sr, r1, r2 = rel(alt['sel'], ref['sel']), rel(alt['t1'], ref['t1']), rel(alt['t2'], ref['t2']); eb_same = bool(np.array_equal(eb_ref, eb_alt))
    return dict(argmin_agree_frac=float(same.mean()), flip_frac=float((~same).mean()), flip_to_antipode_frac=(float(anti[~same].mean()) if (~same).any() else None), min_margin_at_flip=(float(ref['mg'][~same].min()) if (~same).any() else None),
                axis_ok=bool(np.all(axis_ok)), sel_rel=sr, t1_rel=r1, t2_rel=r2, eventB_identical=eb_same, ok=bool(np.all(axis_ok) and sr < sel_tol and r1 < eval_tol and r2 < eval_tol and eb_same))
# self-test of the predicate against synthetic cases (required gate): the report's claims are tested against the SAME function used for the gates
def _mk(am, sel, t1, t2, mg): return dict(a='', am=np.array(am), sel=np.array(sel, float), t1=np.array(t1, float), t2=np.array(t2, float), mg=np.array(mg, float))
_ref = _mk([10, 20, 30], [100., 110., 120.], [50., 51., 52.], [60., 61., 62.], [0.0, 1e-2, 1e-2]); _A10 = int(ANTIPODE[10]); _far = int(next(j for j in range(3072) if j != 10 and j != _A10))
_c1 = selected_output_equiv(_ref, _mk([_A10, 20, 30], [100. * (1 + 5e-7), 110., 120.], [50., 51., 52.], [60., 61., 62.], [0, 0, 0]), 'float32')['ok']        # 1 f32: exact antipode, margin 0, outputs identical -> PASS
_c2 = selected_output_equiv(_ref, _mk([_far, 20, 30], [100., 110., 120.], [50., 51., 52.], [60., 61., 62.], [0, 0, 0]), 'float32')['ok']                  # 2 f32: distant (non-antipodal) axis, margin 0, outputs identical -> FAIL
_c3 = selected_output_equiv(_ref, _mk([_A10, 20, 30], [100., 110., 120.], [50. * 1.001, 51., 52.], [60., 61., 62.], [0, 0, 0]), 'float32')['ok']          # 3 f32: exact antipode but T1 differs > TOL_EVAL -> FAIL
_ref2 = _mk([10, 20, 30], [100., 110., 120.], [50., 51., 52.], [60., 61., 62.], [1e-2, 1e-2, 1e-2])
_c4 = selected_output_equiv(_ref2, _mk([_A10, 20, 30], [100., 110., 120.], [50., 51., 52.], [60., 61., 62.], [0, 0, 0]), 'float32')['ok']                 # 4 f32: exact antipode but margin >= selection tol -> FAIL
_c5 = selected_output_equiv(_ref, _mk([_A10, 20, 30], [100., 110., 120.], [50., 51., 52.], [60., 61., 62.], [0, 0, 0]), 'float64')['ok']                   # 5 f64: any flip (even exact antipode) -> FAIL (raw exact)
_c6 = selected_output_equiv(_ref, _mk([10, 20, 30], [100., 110., 120.], [30., 51., 52.], [60., 61., 62.], [0, 0, 0]), 'float32', eval_tol=1.0)['ok']       # 6 f32: same axes but the Event B indicator changes -> FAIL (independent of eval_tol)
GATES['G_plane_equivalence_gate_selftest'] = bool(_c1 and not _c2 and not _c3 and not _c4 and not _c5 and not _c6)
DIAG['gate_selftest'] = dict(antipode_flip_pass=_c1, distant_flip_fail=(not _c2), antipode_T1_differs_fail=(not _c3), antipode_large_margin_fail=(not _c4), float64_flip_fail=(not _c5), eventB_change_fail=(not _c6))
assert all(GATES.values()), {k: v for k, v in GATES.items() if not v}   # preflight: every cheap gate must pass before the benchmark starts
print('preflight OK:', {k: GATES[k] for k in ('G_parent_failed_smoke_binding', 'G_parent_failed_smoke_artifacts', 'G_antipode_map_involution', 'G_antipode_map_expected_sha', 'G_eventB_thresholds_bound', 'G_plane_equivalence_gate_selftest')})


In [ ]:
# ---- 2. 子プロセス（CPU）：resident copy・thread 固定・失敗分類・selection score ----
CHILD = os.path.join(LOCAL, 'a8b_child.py')
open(CHILD, 'w').write(textwrap.dedent(r'''
import os, sys, json, time, hashlib, threading, resource
cfg = json.loads(sys.argv[1])
for k in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS'): os.environ[k] = str(cfg['threads'])
import numpy as np, psutil
from threadpoolctl import threadpool_info, threadpool_limits
threadpool_limits(limits=cfg['threads'], user_api='blas')
def sha256_file(p, block=8 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(block), b''): h.update(b)
    return h.hexdigest()
LOCAL = cfg['local']; route = cfg['route']; sel_dtype = np.float32 if cfg['dtype'] == 'float32' else np.float64; dtype = sel_dtype; N, CH, AB, SEED = cfg['N'], cfg['chunk'], cfg.get('axis_block', 3072), cfg['seed']   # cfg['dtype'] = SELECTION dtype; evaluation is always float64
proc = psutil.Process(); vm0 = psutil.virtual_memory(); sw0 = psutil.swap_memory().used; avail_before = vm0.available; base_rss = proc.memory_info().rss; peak = {'rss': base_rss, 'phase': 'setup', 'setup': base_rss, 'run': 0}; stop = False; MARKS = {}
ru_start = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024      # inherited high-water mark (fork from the parent): only the delta is meaningful
def mark_rss(name): r = proc.memory_info().rss; peak['rss'] = max(peak['rss'], r); peak[peak['phase']] = max(peak[peak['phase']], r); MARKS[name] = max(MARKS.get(name, 0.0), r / 1e9)
def sampler():
    while not stop:
        r = proc.memory_info().rss; peak['rss'] = max(peak['rss'], r); peak[peak['phase']] = max(peak[peak['phase']], r); time.sleep(0.005)
th = threading.Thread(target=sampler, daemon=True); th.start()
t_setup = time.perf_counter()
fb = os.path.join(LOCAL, 's1_Bstack_l2_4_N16_common_v1.npz'); assert sha256_file(fb) == cfg['expected']['bstack_file']
with np.load(fb) as z: Bp4_64 = np.asarray(z['Bp_stack'], np.float64); Bm4_64 = np.asarray(z['Bm_stack'], np.float64)   # frozen float64 evaluation artifacts
Bp4 = Bp4_64 if sel_dtype == np.float64 else Bp4_64.astype(np.float32)                                                       # selection artifacts
if route.startswith('l24'):
    Bp = Bp4; Bm = Bm4_64; D = 21
    if route == 'l24_feature231':
        iu = np.triu_indices(D); off = (iu[0] != iu[1])
        def fvec(B): v = B[iu].copy(); v[off] *= 2.0; return v
        Fp = np.array([fvec(Bp[a]) for a in range(3072)], dtype=dtype)
else:
    D = 285; iu = np.triu_indices(D); NF = len(iu[0]); ff = os.path.join(LOCAL, 's1_Bplus_featurestack_l2_16_N16_common_v1_%s.npy' % cfg['dtype']); assert sha256_file(ff) == cfg['expected']['F%s_file' % ('64' if cfg['dtype'] == 'float64' else '32')]
    Fm = np.load(ff, mmap_mode='r'); F = np.array(Fm, copy=True, order='C'); assert not np.shares_memory(Fm, F); del Fm; mark_rss('after_F_resident_copy')     # explicit resident copy
setup_s = time.perf_counter() - t_setup; peak['run'] = proc.memory_info().rss; peak['phase'] = 'run'
def packed_into(x, out):
    k = 0; d = x.shape[1]
    for i in range(d):
        w = d - i; np.multiply(x[:, i:i + 1], x[:, i:], out=out[:, k:k + w]); k += w
    return out
def gen(rng, n): return rng.standard_normal((n, D))                      # float64 sample (production keeps float64 for evaluation)
def cast(x64): return x64 if sel_dtype == np.float64 else x64.astype(np.float32)
def select(x):
    # SELECTION (selection dtype): S+ all axes -> argmin (first occurrence) -> selection score; audit mode additionally returns the runner-up
    n = len(x); ar = np.arange(n); AUDIT = cfg.get('audit_mode', False)
    def runner_up(Sp, a):
        Sp2 = Sp.copy(); Sp2[ar, a] = np.inf; return Sp2.min(1)
    if route in ('l24_direct_einsum', 'l24_diagnostic_full_Spm'):
        Sp = np.einsum('ni,aij,nj->na', x, Bp, x, optimize=True); a = Sp.argmin(1); s = Sp[ar, a]
        if route == 'l24_diagnostic_full_Spm': np.einsum('ni,aij,nj->na', x, Bm.astype(sel_dtype), x, optimize=True)   # diagnostic: full S- as well
        return a, s, (runner_up(Sp, a) if AUDIT else None)
    if route == 'l24_feature231':
        f = packed_into(x, np.empty((n, len(iu[0])), sel_dtype)); Sp = f @ Fp.T; a = Sp.argmin(1); return a, Sp[ar, a], (runner_up(Sp, a) if AUDIT else None)
    if route == 's2_feature':
        f = packed_into(x, np.empty((n, NF), sel_dtype)); mark_rss('after_packed_features'); best = np.full(n, np.inf, sel_dtype); a = np.zeros(n, np.int64); second = np.full(n, np.inf, sel_dtype) if AUDIT else None
        for a0 in range(0, 3072, AB):
            S = f @ F[a0:a0 + AB].T; mark_rss('after_score_block'); j = S.argmin(1); v = S[ar, j]; m = v < best
            if AUDIT: S[ar, j] = np.inf; v2 = S.min(1); second = np.where(m, np.minimum(best, v2), np.minimum(second, v))
            best[m] = v[m]; a[m] = (a0 + j)[m]
        return a, best, second
    raise ValueError(route)
def eval64(x64, a):
    # EVALUATION (frozen float64): T1/T2 at the selected axis with float64 x4 and float64 B-stack
    x4 = x64[:, :21]; return np.einsum('ni,nij,nj->n', x4, Bp4_64[a], x4, optimize=True), np.einsum('ni,nij,nj->n', x4, Bm4_64[a], x4, optimize=True)
def run_pass(seed, n_total, keep=0):
    rng = np.random.default_rng(np.random.SeedSequence([seed, 8, 2])); T = dict(draw=0.0, cast=0.0, scan=0.0, eval64=0.0); h = hashlib.sha256(); kept = []
    for i in range(0, n_total, CH):
        n = min(CH, n_total - i)
        t0 = time.perf_counter(); x64 = gen(rng, n); T['draw'] += time.perf_counter() - t0
        t0 = time.perf_counter(); xs = cast(x64); T['cast'] += time.perf_counter() - t0
        t0 = time.perf_counter(); a, s, ru = select(xs); T['scan'] += time.perf_counter() - t0
        t0 = time.perf_counter(); t1, t2 = eval64(x64, a); T['eval64'] += time.perf_counter() - t0
        if not (np.isfinite(s).all() and np.isfinite(t1).all() and np.isfinite(t2).all()): raise FloatingPointError('non-finite output')
        h.update(a.astype(np.int32).tobytes())
        if i < keep: kept.append(np.column_stack([a, s, t1, t2] + ([(ru - s) / np.maximum(np.abs(s), 1e-300)] if ru is not None else []))[:max(0, keep - i)])
    T['production'] = T['cast'] + T['scan'] + T['eval64']; T['end_to_end'] = T['draw'] + T['production']
    return dict(T=T, argmin_hash=h.hexdigest(), audit=(np.vstack(kept) if kept else None))
if cfg.get('audit_mode', False):                                                     # audit child: untimed; processes whole chunks (same GEMM blocking as the timed run) and keeps the first audit_n samples
    n_tot = int(np.ceil(max(cfg['audit_n'], CH) / CH)) * CH; rp = run_pass(SEED, n_tot, keep=cfg['audit_n']); au = rp['audit']
    print(json.dumps(dict(cfg=cfg, status='ok', audit_argmin_sha=hashlib.sha256(au[:, 0].astype(np.int32).tobytes()).hexdigest(), audit_sel=au[:, 1].tolist(), audit_T1=au[:, 2].tolist(), audit_T2=au[:, 3].tolist(), audit_margin=au[:, 4].tolist(), audit_argmin=au[:, 0].astype(int).tolist()))); sys.exit(0)
run_pass(SEED + 1000, min(N, cfg.get('warmup_chunks', 2) * CH))                   # warm-up: a few chunks only
reps = [run_pass(SEED, N, keep=(cfg.get('audit_n', 0) if r == 0 else 0)) for r in range(cfg.get('repeats', 1))]
stop = True; th.join(timeout=1.0); ru = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024; sw1 = psutil.swap_memory().used; mark_rss('end')
peak_eff = max(peak['rss'], proc.memory_info().rss, max(MARKS.values()) * 1e9)          # sampled (5 ms) + explicit markers; ru_maxrss is inherited across fork and kept as a delta diagnostic only
peak_run = max(peak['run'], max([v for k, v in MARKS.items() if k != 'after_F_resident_copy'] or [0]) * 1e9)
ru_delta = max(0.0, ru - ru_start)
med = lambda k: float(np.median([r['T'][k] for r in reps]))
out = dict(cfg=cfg, status='ok', setup_s=setup_s, draw_s_median=med('draw'), cast_s_median=med('cast'), scan_s_median=med('scan'), eval64_s_median=med('eval64'), production_s_median=med('production'), end_to_end_s_median=med('end_to_end'),
           production_s_min=float(min(r['T']['production'] for r in reps)), production_s_max=float(max(r['T']['production'] for r in reps)), argmin_hash=reps[0]['argmin_hash'],
           hashes_consistent=len({r['argmin_hash'] for r in reps}) == 1, base_rss_GB=base_rss / 1e9, peak_rss_sampled_GB=peak['rss'] / 1e9, peak_rss_GB=peak_eff / 1e9, peak_increment_GB=(peak_eff - base_rss) / 1e9, peak_setup_GB=peak['setup'] / 1e9, peak_run_increment_GB=(peak_run - base_rss) / 1e9, ru_maxrss_GB=ru / 1e9, ru_maxrss_inherited_at_start_GB=ru_start / 1e9, ru_maxrss_delta_GB=ru_delta / 1e9, rss_markers_GB=MARKS,
           avail_before_setup_GB=avail_before / 1e9, swap_increase_GB=max(0, sw1 - sw0) / 1e9, threadpools=[{k: v for k, v in t.items() if k in ('internal_api', 'num_threads')} for t in threadpool_info()])
au = reps[0]['audit']
if au is not None: out.update(audit_argmin_sha=hashlib.sha256(au[:, 0].astype(np.int32).tobytes()).hexdigest(), audit_sel=au[:, 1].tolist(), audit_T1=au[:, 2].tolist(), audit_T2=au[:, 3].tolist(), audit_argmin=au[:, 0].astype(int).tolist())
print(json.dumps(out))
'''))
CHILD_SHA = sha256_file(CHILD)
def run_child(cfg, timeout):
    cfg = dict(cfg, local=LOCAL, expected=EXPECTED, threads=THREADS)
    try: r = subprocess.run([sys.executable, CHILD, json.dumps(cfg)], capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired: return dict(cfg=cfg, status='timeout')
    if r.returncode != 0:
        err = r.stderr[-1200:]; st = ('oom' if ('MemoryError' in err or 'Cannot allocate' in err or r.returncode in (-9, 137)) else 'killed' if r.returncode < 0 else 'numerical_fail' if 'FloatingPointError' in err else 'other_error')
        return dict(cfg=cfg, status=st, returncode=r.returncode, stderr=err)
    try: return json.loads(r.stdout.strip().splitlines()[-1])
    except Exception as e: return dict(cfg=cfg, status='other_error', stderr=f'unparseable stdout: {e}')
def key_of(cfg): return json.dumps(dict(route=cfg['route'], dtype=cfg['dtype'], N=cfg['N'], chunk=cfg['chunk'], axis_block=cfg.get('axis_block'), stage=cfg['stage']), sort_keys=True)
print('child ready', CHILD_SHA[:12], '/ threads', THREADS)

In [ ]:
# ---- 3. staged benchmark（登録 grid）----
SEED = 20260908; AUDIT_N = 10_000; TIMEOUT = 3600; BUDGET_S2_1E6_S = 40 * 60
DTYPES = ['float64'] + (['float32'] if FLOAT32_ELIGIBLE else [])
if A8_MODE == 'official':
    L24_CHUNKS, S2_CHUNKS, S2_AB = [2000, 5000, 10000, 20000], [64, 128, 256, 512, 1024, 2000], [512, 3072]; N_PILOT, N_FINAL, N_WIN = dict(l24=40_000, s2=10_000), 100_000, dict(l24=1_000_000, s2=200_000); TOPK = dict(l24=2, s2=3)   # pilot N >= 2 x max chunk
else:
    L24_CHUNKS, S2_CHUNKS, S2_AB = [1000, 2000], [128, 256], [3072]; N_PILOT, N_FINAL, N_WIN = dict(l24=4000, s2=512), 1000, dict(l24=2000, s2=512); TOPK = dict(l24=2, s2=2); AUDIT_N = 500; TIMEOUT = 900
REGISTERED_PILOT = []
for dt in DTYPES:
    for route in ('l24_direct_einsum', 'l24_feature231'):
        for ch in L24_CHUNKS: REGISTERED_PILOT.append(dict(route=route, dtype=dt, N=N_PILOT['l24'], chunk=ch, stage='pilot'))
    for ch in S2_CHUNKS:
        for ab in S2_AB: REGISTERED_PILOT.append(dict(route='s2_feature', dtype=dt, N=N_PILOT['s2'], chunk=ch, axis_block=ab, stage='pilot'))
REGISTERED_PILOT.append(dict(route='l24_diagnostic_full_Spm', dtype='float64', N=N_PILOT['l24'], chunk=L24_CHUNKS[-1], stage='pilot'))
def base(cfg): return dict(cfg, seed=SEED, warmup_chunks=2)
rows = []; t_all = time.time()
BENCH_ENV = dict(python=ENV['python'], numpy=ENV['numpy'], platform=ENV['platform'], cpu_model=ENV['cpu_model'], affinity=THREADS, threadpools=sorted([(t['internal_api'], t.get('version')) for t in ENV['threadpools_parent']]),
                 a8a_provenance_sha=A8A_PROV_SHA, manifest_sha=a8a['manifest_file_sha256'], float32_event=F32_EVENT, float32_axis=F32_AXIS,
                 ram_total_bytes=int(psutil.virtual_memory().total), swap_total_bytes=int(psutil.swap_memory().total),
                 cgroup_memory_limit=next((open(p).read().strip() for p in ('/sys/fs/cgroup/memory.max', '/sys/fs/cgroup/memory/memory.limit_in_bytes') if os.path.exists(p)), None))
BINDING = hashlib.sha256(json.dumps(dict(nb=NB_HEAD_SHA, child=CHILD_SHA, expected=EXPECTED, threads=THREADS, seed=SEED, env=BENCH_ENV), sort_keys=True).encode()).hexdigest()
RUN_ID = BINDING[:12]; CKPT = os.path.join(OUT, 'attempts', RUN_ID); os.makedirs(CKPT, exist_ok=True)   # environment-specific run id: checkpoints from a different runtime are never mixed
def atomic_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as fh: json.dump(obj, fh); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path)
def run_stage(cfgs, repeats, audit):
    for g in cfgs:
        ck = os.path.join(CKPT, hashlib.sha256((key_of(g) + BINDING).encode()).hexdigest()[:24] + '.json')
        if os.path.exists(ck):
            r = json.load(open(ck))
            if r.get('binding') == BINDING and r.get('registered_key') == key_of(g): rows.append(r); print(f"[{g['stage']:8s}] resumed {g['route']} {g['dtype']} ch={g['chunk']}"); continue
        r = run_child(dict(base(g), repeats=repeats, audit_n=(AUDIT_N if (audit and g['N'] >= AUDIT_N) else 0)), TIMEOUT); r['registered_key'] = key_of(g); r['binding'] = BINDING; rows.append(r)
        atomic_json(r, ck)
        if r['status'] == 'ok': print(f"[{g['stage']:8s}] {g['route']:24s} sel={g['dtype']:7s} N={g['N']:>8d} ch={g['chunk']:>5d} ab={g.get('axis_block','-')!s:>4s} production {r['production_s_median']:8.2f}s (scan {r['scan_s_median']:.2f} eval64 {r['eval64_s_median']:.2f}) +{r['peak_increment_GB']:.2f}GB")
        else: print(f"[{g['stage']:8s}] {g['route']:24s} {g['dtype']:7s} N={g['N']:>8d} ch={g['chunk']:>5d} -> {r['status']}")
def eligible(r): return r['status'] == 'ok' and r['hashes_consistent'] and r['peak_increment_GB'] <= 0.65 * r['avail_before_setup_GB'] and r['swap_increase_GB'] == 0
def thr(r): return r['cfg']['N'] / r['production_s_median']          # selection metric: production time (cast + scan + eval64)
def cands(route, dtype, stage): return [r for r in rows if r['cfg']['route'] == route and r['cfg']['dtype'] == dtype and r['cfg']['stage'] == stage and eligible(r)]
def safe_choice(c):                                   # registered: max throughput; within 5% -> smallest (chunk, axis_block)
    if not c: return None
    best = max(thr(r) for r in c); band = [r for r in c if thr(r) >= 0.95 * best]; return min(band, key=lambda r: (r['cfg']['chunk'], r['cfg'].get('axis_block') or 0))
def top(route, dtype, stage, k):
    c = cands(route, dtype, stage); raw = sorted(c, key=lambda r: (-thr(r), r['cfg']['chunk'], r['cfg'].get('axis_block') or 0))[:k]; sc = safe_choice(c)
    if sc and sc not in raw: raw.append(sc)          # always keep the safe-choice config among finalists
    return raw
def winner(route, dtype, stage): sc = safe_choice(cands(route, dtype, stage)); return [sc] if sc else []
run_stage(REGISTERED_PILOT, repeats=1, audit=True)
REGISTERED_FINAL = []
for dt in DTYPES:
    for route in ('l24_direct_einsum', 'l24_feature231'):
        for r in top(route, dt, 'pilot', TOPK['l24']): REGISTERED_FINAL.append(dict(route=route, dtype=dt, N=N_FINAL, chunk=r['cfg']['chunk'], stage='finalist'))
    for r in top('s2_feature', dt, 'pilot', TOPK['s2']): REGISTERED_FINAL.append(dict(route='s2_feature', dtype=dt, N=N_FINAL, chunk=r['cfg']['chunk'], axis_block=r['cfg']['axis_block'], stage='finalist'))
run_stage(REGISTERED_FINAL, repeats=3, audit=True)
REGISTERED_WIN = []
for dt in DTYPES:
    for route in ('l24_direct_einsum', 'l24_feature231'):
        for r in winner(route, dt, 'finalist'): REGISTERED_WIN.append(dict(route=route, dtype=dt, N=N_WIN['l24'], chunk=r['cfg']['chunk'], stage='winner'))
    for r in winner('s2_feature', dt, 'finalist'): REGISTERED_WIN.append(dict(route='s2_feature', dtype=dt, N=N_WIN['s2'], chunk=r['cfg']['chunk'], axis_block=r['cfg']['axis_block'], stage='winner'))
run_stage(REGISTERED_WIN, repeats=3, audit=True)
# optional S2 N=1e6 (registered rule)
OPT = []
if A8_MODE == 'official':
    s2w = [r for r in rows if r['cfg']['route'] == 's2_feature' and r['cfg']['stage'] == 'winner' and eligible(r)]
    if s2w:
        best = safe_choice(s2w); ps = {}
        for st in ('pilot', 'finalist', 'winner'):
            m = [r for r in rows if r['cfg']['route'] == 's2_feature' and r['cfg']['dtype'] == best['cfg']['dtype'] and r['cfg']['chunk'] == best['cfg']['chunk'] and r['cfg'].get('axis_block') == best['cfg'].get('axis_block') and r['cfg']['stage'] == st and r['status'] == 'ok']
            if m: ps[st] = m[0]['production_s_median'] / m[0]['cfg']['N']
        linear_ok = (max(ps.values()) / min(ps.values()) < 1.2) if len(ps) >= 2 else False; projected = (best['production_s_median'] / best['cfg']['N']) * 1e6
        DIAG['s2_1e6'] = dict(best_cfg={k: v for k, v in best['cfg'].items() if k in ('route', 'dtype', 'chunk', 'axis_block')}, per_sample_by_stage=ps, linearity_ok=linear_ok, projected_s=projected, budget_s=BUDGET_S2_1E6_S, measured=False)
        if linear_ok and projected < BUDGET_S2_1E6_S:
            g = dict(route='s2_feature', dtype=best['cfg']['dtype'], N=1_000_000, chunk=best['cfg']['chunk'], axis_block=best['cfg']['axis_block'], stage='optional_1e6'); OPT.append(g)
            ck = os.path.join(CKPT, hashlib.sha256((key_of(g) + BINDING).encode()).hexdigest()[:24] + '.json')
            if os.path.exists(ck) and json.load(open(ck)).get('binding') == BINDING: r = json.load(open(ck))
            else: r = run_child(dict(base(g), repeats=1, audit_n=0), TIMEOUT * 2); r['registered_key'] = key_of(g); r['binding'] = BINDING; atomic_json(r, ck)
            rows.append(r); DIAG['s2_1e6']['measured'] = (r['status'] == 'ok')
print(f'CPU stages done {(time.time()-t_all)/60:.1f} min')

In [ ]:
# ---- 4. GPU（strict fp32・TF32 診断）：generation / H2D / kernel / D2H を分離 ----
GPU_CHILD = os.path.join(LOCAL, 'a8b_gpu_child.py')
open(GPU_CHILD, 'w').write(textwrap.dedent(r'''
import os, sys, json, time, hashlib
cfg = json.loads(sys.argv[1]); LOCAL = cfg['local']
for k in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS'): os.environ[k] = str(cfg['threads'])     # host float64 evaluation threads fixed BEFORE numpy import
import numpy as np
from threadpoolctl import threadpool_info, threadpool_limits
threadpool_limits(limits=cfg['threads'], user_api='blas')
def sha256_file(p, block=8 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(block), b''): h.update(b)
    return h.hexdigest()
try:
    import torch
    if not torch.cuda.is_available(): print(json.dumps(dict(cfg=cfg, status='unavailable'))); sys.exit(0)
    if cfg.get('probe_only'): print(json.dumps(dict(cfg=cfg, status='ok', gpu=torch.cuda.get_device_name(0), vram_total_GB=torch.cuda.get_device_properties(0).total_memory / 1e9, torch=torch.__version__, cuda=torch.version.cuda))); sys.exit(0)
    tf32 = bool(cfg['tf32']); torch.backends.cuda.matmul.allow_tf32 = tf32; torch.backends.cudnn.allow_tf32 = tf32; torch.set_float32_matmul_precision('high' if tf32 else 'highest')
    N, CH, AB, SEED = cfg['N'], cfg['chunk'], cfg.get('axis_block', 3072), cfg['seed']; D = 285; iu = np.triu_indices(D)
    ff = os.path.join(LOCAL, 's1_Bplus_featurestack_l2_16_N16_common_v1_float32.npy'); assert sha256_file(ff) == cfg['expected']['F32_file']
    t0 = time.perf_counter(); F = torch.from_numpy(np.load(ff)).cuda(); i0 = torch.from_numpy(iu[0]).cuda(); i1 = torch.from_numpy(iu[1]).cuda()
    fb = os.path.join(LOCAL, 's1_Bstack_l2_4_N16_common_v1.npz'); assert sha256_file(fb) == cfg['expected']['bstack_file']
    with np.load(fb) as z: Bp4_64 = np.asarray(z['Bp_stack'], np.float64); Bm4_64 = np.asarray(z['Bm_stack'], np.float64)   # host float64 evaluation
    torch.cuda.synchronize(); setup_s = time.perf_counter() - t0
    def kernel(x):   # NOTE: GPU route builds the CH x D x D outer product on device (memory algorithm differs from the CPU packed route)
        n = x.shape[0]; f = (x[:, :, None] * x[:, None, :])[:, i0, i1]; best = torch.full((n,), float('inf'), device='cuda'); a = torch.zeros(n, dtype=torch.long, device='cuda')
        for a0 in range(0, 3072, AB):
            S = f @ F[a0:a0 + AB].T; v, j = S.min(1); m = v < best; best[m] = v[m]; a[m] = (a0 + j)[m]
        return a, best
    def run_pass(seed, n_total, keep=0):
        rng = np.random.default_rng(np.random.SeedSequence([seed, 8, 2])); h = hashlib.sha256(); kept = []; T = dict(draw=0.0, cast=0.0, h2d=0.0, scan=0.0, d2h=0.0, eval64=0.0)
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
        for i in range(0, n_total, CH):
            n = min(CH, n_total - i); t0 = time.perf_counter(); x64 = rng.standard_normal((n, D)); T['draw'] += time.perf_counter() - t0
            t0 = time.perf_counter(); x32 = x64.astype(np.float32); T['cast'] += time.perf_counter() - t0
            t0 = time.perf_counter(); x = torch.from_numpy(x32).cuda(); torch.cuda.synchronize(); T['h2d'] += time.perf_counter() - t0
            t0 = time.perf_counter(); a, s = kernel(x); torch.cuda.synchronize(); T['scan'] += time.perf_counter() - t0
            t0 = time.perf_counter(); an = a.cpu().numpy(); sn = s.cpu().numpy(); T['d2h'] += time.perf_counter() - t0
            t0 = time.perf_counter(); x4 = x64[:, :21]; t1n = np.einsum('ni,nij,nj->n', x4, Bp4_64[an], x4, optimize=True); t2n = np.einsum('ni,nij,nj->n', x4, Bm4_64[an], x4, optimize=True); T['eval64'] += time.perf_counter() - t0   # frozen float64 evaluation on host
            h.update(an.astype(np.int32).tobytes())
            if i < keep: kept.append(np.column_stack([an, sn, t1n, t2n])[:max(0, keep - i)])
        T['production'] = T['cast'] + T['h2d'] + T['scan'] + T['d2h'] + T['eval64']; T['end_to_end'] = T['draw'] + T['production']
        return dict(T=T, argmin_hash=h.hexdigest(), audit=(np.vstack(kept) if kept else None), peak_alloc_GB=torch.cuda.max_memory_allocated() / 1e9, peak_reserved_GB=torch.cuda.max_memory_reserved() / 1e9)
    run_pass(SEED + 1000, min(N, 2 * CH)); reps = [run_pass(SEED, N, keep=(cfg.get('audit_n', 0) if r == 0 else 0)) for r in range(cfg.get('repeats', 1))]
    med = lambda k: float(np.median([r['T'][k] for r in reps]))
    out = dict(cfg=cfg, status='ok', setup_s=setup_s, draw_s_median=med('draw'), cast_s_median=med('cast'), h2d_s_median=med('h2d'), scan_s_median=med('scan'), d2h_s_median=med('d2h'), eval64_s_median=med('eval64'),
               production_s_median=med('production'), end_to_end_s_median=med('end_to_end'), argmin_hash=reps[0]['argmin_hash'], hashes_consistent=len({r['argmin_hash'] for r in reps}) == 1,
               peak_alloc_GB=max(r['peak_alloc_GB'] for r in reps), peak_reserved_GB=max(r['peak_reserved_GB'] for r in reps), vram_total_GB=torch.cuda.get_device_properties(0).total_memory / 1e9, torch=torch.__version__, cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0),
               threadpools=[{k: v for k, v in t.items() if k in ('internal_api', 'num_threads')} for t in threadpool_info()])
    au = reps[0]['audit']
    if au is not None: out.update(audit_argmin_sha=hashlib.sha256(au[:, 0].astype(np.int32).tobytes()).hexdigest(), audit_sel=au[:, 1].tolist(), audit_T1=au[:, 2].tolist(), audit_T2=au[:, 3].tolist(), audit_argmin=au[:, 0].astype(int).tolist())
    print(json.dumps(out))
except Exception as e:
    st = 'oom' if 'out of memory' in str(e).lower() else 'available_but_failed'; print(json.dumps(dict(cfg=cfg, status=st, error=f'{type(e).__name__}: {str(e)[:300]}')))
'''))
GPU_CHILD_SHA = sha256_file(GPU_CHILD)
def _gpu_exec(cfg, timeout):
    try: r = subprocess.run([sys.executable, GPU_CHILD, json.dumps(cfg)], capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired: return dict(cfg=cfg, status='timeout')
    if r.returncode != 0 or not r.stdout.strip(): return dict(cfg=cfg, status='killed' if r.returncode < 0 else 'other_error', returncode=r.returncode, stderr=r.stderr[-800:])
    return json.loads(r.stdout.strip().splitlines()[-1])
GPU_BINDING = None
def run_gpu(cfg, timeout=7200):
    cfg = dict(cfg, local=LOCAL, expected=EXPECTED, seed=SEED, threads=THREADS); assert GPU_BINDING is not None, 'GPU probe must establish GPU_BINDING first'
    ck = os.path.join(CKPT, 'gpu_' + hashlib.sha256((json.dumps(cfg, sort_keys=True) + GPU_BINDING).encode()).hexdigest()[:24] + '.json')
    if os.path.exists(ck):
        old = json.load(open(ck))
        if old.get('gpu_binding') == GPU_BINDING and all(old.get(k) == PROBE.get(k) for k in ('gpu', 'torch', 'cuda') if old.get('status') == 'ok'): return old
    out = _gpu_exec(cfg, timeout); out['gpu_binding'] = GPU_BINDING; atomic_json(out, ck); return out
gpu_rows = []; GPU_STATUS = 'unavailable'; PROBE = {}
if A8_MODE == 'official':
    PROBE = _gpu_exec(dict(route='s2_feature_gpu', dtype='float32', tf32=False, N=1000, chunk=256, axis_block=3072, repeats=1, audit_n=0, stage='probe', probe_only=True, local=LOCAL, expected=EXPECTED, seed=SEED, threads=THREADS), 600)   # always uncached
    if PROBE.get('status') == 'ok':
        GPU_BINDING = hashlib.sha256((BINDING + json.dumps(dict(gpu=PROBE['gpu'], vram=PROBE['vram_total_GB'], torch=PROBE['torch'], cuda=PROBE['cuda'], gpu_child=GPU_CHILD_SHA, nvidia_smi=ENV['gpu']), sort_keys=True)).encode()).hexdigest()
    probe = PROBE
    if probe.get('status') == 'ok':
        for tf32 in (False, True):
            for ch in (256, 1024, 4000): gpu_rows.append(run_gpu(dict(route='s2_feature_gpu', dtype='float32', tf32=tf32, N=10_000, chunk=ch, axis_block=3072, repeats=1, audit_n=AUDIT_N, stage='pilot')))
            okp = [r for r in gpu_rows if r['cfg']['tf32'] == tf32 and r['cfg']['stage'] == 'pilot' and r['status'] == 'ok' and r['peak_reserved_GB'] <= 0.65 * r['vram_total_GB']]
            if okp:
                best = max(r['cfg']['N'] / r['production_s_median'] for r in okp); band = [r for r in okp if r['cfg']['N'] / r['production_s_median'] >= 0.95 * best]; b = min(band, key=lambda r: r['cfg']['chunk'])   # safe tie-break
                gpu_rows.append(run_gpu(dict(route='s2_feature_gpu', dtype='float32', tf32=tf32, N=100_000, chunk=b['cfg']['chunk'], axis_block=3072, repeats=3, audit_n=AUDIT_N, stage='finalist')))
        gw = [r for r in gpu_rows if r['status'] == 'ok' and r['cfg']['stage'] == 'finalist' and not r['cfg']['tf32'] and r['peak_reserved_GB'] <= 0.65 * r['vram_total_GB']]
        if gw:
            b = gw[0]; pil = [r for r in gpu_rows if r['status'] == 'ok' and r['cfg']['stage'] == 'pilot' and not r['cfg']['tf32'] and r['cfg']['chunk'] == b['cfg']['chunk']]
            ps = [r['production_s_median'] / r['cfg']['N'] for r in pil + [b]]; lin = (max(ps) / min(ps) < 1.2) if len(ps) >= 2 else False; proj = b['production_s_median'] / b['cfg']['N'] * 1e6
            DIAG['gpu_1e6'] = dict(linearity_ok=lin, projected_s=proj, budget_s=BUDGET_S2_1E6_S, measured=False)
            if lin and proj < BUDGET_S2_1E6_S: gpu_rows.append(run_gpu(dict(route='s2_feature_gpu', dtype='float32', tf32=False, N=1_000_000, chunk=b['cfg']['chunk'], axis_block=3072, repeats=1, audit_n=0, stage='optional_1e6'))); DIAG['gpu_1e6']['measured'] = (gpu_rows[-1]['status'] == 'ok')
        GPU_STATUS = 'ok' if any(r['status'] == 'ok' for r in gpu_rows) else 'available_but_failed'
    elif probe.get('status') == 'unavailable': GPU_STATUS = 'unavailable'
    else: GPU_STATUS = 'available_but_failed'
DIAG['gpu_probe'] = {k: v for k, v in PROBE.items() if k != 'cfg'}; DIAG['gpu_binding'] = GPU_BINDING
for r in gpu_rows: print('GPU', r['cfg']['stage'], 'tf32=%s' % r['cfg']['tf32'], 'N=%d ch=%d' % (r['cfg']['N'], r['cfg']['chunk']), r['status'], 'production', r.get('production_s_median', ''), 'peak_alloc', r.get('peak_alloc_GB', ''))
print('GPU status:', GPU_STATUS)

In [ ]:
# ---- 5. gate・cross-route・ROUTE_DECISION・provenance ----
ALLOWED = {'ok', 'oom', 'timeout', 'killed', 'numerical_fail', 'other_error'}
reg_keys = [key_of(g) for g in REGISTERED_PILOT + REGISTERED_FINAL + REGISTERED_WIN]; att_keys = [r['registered_key'] for r in rows if r['cfg']['stage'] != 'optional_1e6']
GATES['G_registered_inventory_exact'] = (sorted(reg_keys) == sorted(att_keys) and len(set(att_keys)) == len(att_keys))
GATES['G_all_attempts_classified'] = all(r['status'] in ALLOWED for r in rows)
okr = [r for r in rows if r['status'] == 'ok']
GATES['G_all_ok_timings_finite_positive'] = all(all(np.isfinite(r[f'{k}_s_median']) and r[f'{k}_s_median'] >= 0 for k in ('draw', 'cast', 'scan', 'eval64', 'production', 'end_to_end')) and r['production_s_median'] > 0 and r['scan_s_median'] > 0 and r['hashes_consistent'] for r in okr)
GATES['G_memory_telemetry_complete'] = all(all(k in r for k in ('base_rss_GB', 'peak_rss_GB', 'peak_rss_sampled_GB', 'peak_increment_GB', 'peak_setup_GB', 'peak_run_increment_GB', 'ru_maxrss_GB', 'ru_maxrss_delta_GB', 'avail_before_setup_GB', 'swap_increase_GB', 'rss_markers_GB')) for r in okr)
def registered_threads_ok(r):
    pools = [p for p in r['threadpools'] if p['internal_api'] in {'openblas', 'mkl', 'blis'}]; return bool(pools) and all(p['num_threads'] == THREADS for p in pools)   # every BLAS pool entry
GATES['G_thread_configuration_registered'] = all(registered_threads_ok(r) for r in okr) if okr else False
DIAG['thread_configuration_constant'] = (len({json.dumps(sorted([(p['internal_api'], p['num_threads']) for p in r['threadpools']])) for r in okr}) == 1) if okr else False
GATES['G_feasible_config_per_required_route'] = all(any(eligible(r) and r['cfg']['route'] == route and r['cfg']['dtype'] == dt for r in okr) for route in ('l24_direct_einsum', 'l24_feature231', 's2_feature') for dt in DTYPES)
# untimed audit children: one per (route, dtype) at the winner (or best pilot) config; provide true top-two margins and must reproduce the timed audit vectors
AUDITS = {}
for key in {(r['cfg']['route'], r['cfg']['dtype']) for r in okr}:
    ref = ([r for r in okr if (r['cfg']['route'], r['cfg']['dtype']) == key and r['cfg']['stage'] == 'winner'] or [r for r in okr if (r['cfg']['route'], r['cfg']['dtype']) == key])[0]
    g = dict(ref['cfg']); g.update(stage='audit', audit_mode=True, audit_n=AUDIT_N, N=AUDIT_N); a = run_child(dict(base(g), repeats=1), TIMEOUT)
    if a.get('status') == 'ok': AUDITS[key] = a
GATES['G_audit_children_ok'] = all(k in AUDITS for k in {(r['cfg']['route'], r['cfg']['dtype']) for r in okr})
def audit_matches(k):
    # every timed row with the SAME (chunk, axis_block) as the untimed audit child must be selected-output-equivalent to it (audit mode is non-invasive); one shared predicate (v1.1.7)
    A = AUDITS[k]; ref = au(A); rows_same = [r for r in okr if (r['cfg']['route'], r['cfg']['dtype']) == k and 'audit_argmin_sha' in r and r['cfg']['chunk'] == A['cfg']['chunk'] and r['cfg'].get('axis_block') == A['cfg'].get('axis_block')]
    return bool(rows_same) and all(selected_output_equiv(ref, au(r), k[1])['ok'] for r in rows_same)
with open(os.path.join(OUT, 'a8b_audit_vectors.npz.tmp'), 'wb') as _fh: np.savez(_fh, **{f"{k[0]}__{k[1]}__{name}": np.array(v[f'audit_{name}']) for k, v in AUDITS.items() for name in ('argmin', 'sel', 'T1', 'T2', 'margin')})
os.replace(os.path.join(OUT, 'a8b_audit_vectors.npz.tmp'), os.path.join(OUT, 'a8b_audit_vectors.npz'))
# timed audit vectors (v1.1.8): every timed CPU/GPU row that carries an audit subset is archived (argmin / selection score / T1 / T2 [0:AUDIT_N]) so that
# G_audit_matches_timed, G_chunk_invariance, G_float64_raw_argmin_exact and the exact-antipode flip accounting can be recomputed by a third party from the freeze
TA_INDEX = []; TA_ARR = {}
for j, r in enumerate([x for x in rows if 'audit_argmin_sha' in x] + [x for x in gpu_rows if 'audit_argmin_sha' in x]):
    cid = f'cfg{j:03d}'; cg = r['cfg']
    TA_INDEX.append(dict(config_id=cid, route=cg['route'], selection_dtype=cg['dtype'], stage=cg['stage'], N=cg['N'], chunk=cg['chunk'], axis_block=cg.get('axis_block'), tf32=cg.get('tf32'), registered_key=r.get('registered_key'), argmin_hash=r.get('argmin_hash'), audit_argmin_sha=r['audit_argmin_sha'], n_audit=len(r['audit_argmin'])))
    TA_ARR[f'{cid}__argmin'] = np.array(r['audit_argmin'], np.int32); TA_ARR[f'{cid}__sel'] = np.array(r['audit_sel'], np.float64); TA_ARR[f'{cid}__T1'] = np.array(r['audit_T1'], np.float64); TA_ARR[f'{cid}__T2'] = np.array(r['audit_T2'], np.float64)
with open(os.path.join(OUT, 'a8b_timed_audit_vectors.npz.tmp'), 'wb') as _fh: np.savez_compressed(_fh, **TA_ARR)
os.replace(os.path.join(OUT, 'a8b_timed_audit_vectors.npz.tmp'), os.path.join(OUT, 'a8b_timed_audit_vectors.npz')); atomic_json(dict(audit_n=AUDIT_N, canonical_reference='a8b_audit_vectors.npz (untimed audit child per route/dtype, carries raw_oriented_axis_margin)', configs=TA_INDEX), os.path.join(OUT, 'a8b_timed_audit_index.json'))
with np.load(os.path.join(OUT, 'a8b_timed_audit_vectors.npz')) as _z:                                              # reopen: every archived vector must equal the in-memory one
    GATES['G_timed_audit_vectors_saved'] = bool(TA_INDEX) and set(_z.keys()) == set(TA_ARR) and all(np.array_equal(_z[k], v) for k, v in TA_ARR.items()) and all(_z[f"{e['config_id']}__argmin"].shape[0] == e['n_audit'] for e in TA_INDEX)
GATES['G_audit_matches_timed'] = all(audit_matches(k) for k in AUDITS)      # audit mode must not change the selected outputs (shared predicate)
# chunk/axis-block invariance within (route, dtype): canonical reference = the untimed audit child (reference axis, raw margin and selected outputs come from one computation); every audited timed config must be equivalent
inv = {}
for key in {(r['cfg']['route'], r['cfg']['dtype']) for r in okr}:
    grp = [r for r in okr if (r['cfg']['route'], r['cfg']['dtype']) == key and 'audit_argmin_sha' in r]
    if key not in AUDITS or not grp: continue
    ref = au(AUDITS[key]); res = []
    for r in grp:
        e = selected_output_equiv(ref, au(r), key[1]); e['cfg'] = {k: v for k, v in r['cfg'].items() if k in ('chunk', 'axis_block', 'stage')}; res.append(e)
    inv[f'{key[0]}/{key[1]}'] = dict(n=len(grp), distinct_configs=len({(r['cfg']['chunk'], r['cfg'].get('axis_block')) for r in grp}), all_ok=all(x['ok'] for x in res), any_flip=any(x['flip_frac'] > 0 for x in res),
                                    all_flips_antipodal=all((x['flip_to_antipode_frac'] in (None, 1.0)) for x in res), detail=res)
GATES['G_chunk_invariance'] = bool(inv) and all(v['all_ok'] for v in inv.values())
GATES['G_chunk_invariance_coverage'] = all(inv.get(f'{route}/{dt}', {}).get('distinct_configs', 0) >= 2 for route in ('l24_direct_einsum', 'l24_feature231', 's2_feature') for dt in DTYPES)   # >= 2 DISTINCT (chunk, axis_block) per production candidate
GATES['G_float64_raw_argmin_exact'] = bool([k for k in inv if k.endswith('/float64')]) and all(all(x['argmin_agree_frac'] == 1.0 for x in v['detail']) for k, v in inv.items() if k.endswith('/float64'))
# cross-route / cross-dtype / GPU-vs-CPU: the same predicate. B = reference (carries the margins), A = alternative; float64 rule only when both sides are float64
def first_audit(route, dtype, rowsrc=None):
    a = AUDITS.get((route, dtype)); return au(a) if a else None
def cross(A, B, t): return selected_output_equiv(B, A, ('float64' if t <= 1e-9 else 'float32'), sel_tol=t)
XR = {}
for dt in DTYPES:
    A, B = first_audit('l24_direct_einsum', dt), first_audit('l24_feature231', dt)
    if A and B: XR[f'l24_direct_vs_feature/{dt}'] = cross(A, B, TOL[dt])
if 'float32' in DTYPES:
    for route in ('l24_direct_einsum', 'l24_feature231', 's2_feature'):
        A, B = first_audit(route, 'float32'), first_audit(route, 'float64')
        if A and B: XR[f'{route}/f32_vs_f64'] = cross(A, B, 1e-4)
GATES['G_l24_routes_agree_f64'] = bool(XR.get('l24_direct_vs_feature/float64', {}).get('ok', False))
GATES['G_l24_routes_agree_f32'] = bool(XR.get('l24_direct_vs_feature/float32', {}).get('ok', False)) if 'float32' in DTYPES else True
gok = [r for r in gpu_rows if r['status'] == 'ok' and 'audit_argmin_sha' in r]
if gok:
    Bc = first_audit('s2_feature', 'float32') if 'float32' in DTYPES else first_audit('s2_feature', 'float64')
    for r in gok:
        if Bc: XR[f"s2_gpu_{'tf32' if r['cfg']['tf32'] else 'strict'}_{r['cfg']['stage']}_ch{r['cfg']['chunk']}_vs_cpu"] = cross(au(r), Bc, 1e-4)
GATES['G_gpu_classified'] = (GPU_STATUS in ('ok', 'unavailable', 'available_but_failed'))
# ---- ROUTE_DECISION (registered rules) ----
def route_dtype(route):  # route-specific dtype eligibility: float32 only if A8a eligible AND this route's f32-vs-f64 cross-check passed
    return 'float32' if ('float32' in DTYPES and XR.get(f'{route}/f32_vs_f64', {}).get('ok')) else 'float64'
def pick(route):
    dtype = route_dtype(route); c = cands(route, dtype, 'winner'); r = safe_choice(c)
    if r is None: return None
    opt = [x for x in okr if x['cfg']['route'] == route and x['cfg']['dtype'] == dtype and x['cfg']['stage'] == 'optional_1e6' and x['cfg']['chunk'] == r['cfg']['chunk'] and x['cfg'].get('axis_block') == r['cfg'].get('axis_block') and eligible(x)]
    src = opt[0] if opt else r
    stage_rank = {'winner': 2, 'finalist': 1, 'pilot': 0, 'optional_1e6': 3}; reps = {}
    for x in okr:
        if not (x['cfg']['route'] == route and x['cfg']['dtype'] == dtype and eligible(x)): continue
        k = (x['cfg']['chunk'], x['cfg'].get('axis_block'))
        if k[0] < r['cfg']['chunk'] or (k[0] == r['cfg']['chunk'] and (k[1] or 0) < (r['cfg'].get('axis_block') or 0)):
            if k not in reps or (x['cfg']['N'], stage_rank[x['cfg']['stage']]) > (reps[k]['cfg']['N'], stage_rank[reps[k]['cfg']['stage']]): reps[k] = x   # representative: max N, then stage priority
    fb = sorted(reps.items(), key=lambda kv: (-kv[0][0], kv[0][1] or 0))[:3]; fb = [(k[0], k[1], v['cfg']['N']) for k, v in fb]
    return dict(route=route, selection_dtype=dtype, evaluation_dtype='float64', sample_chunk=r['cfg']['chunk'], axis_block=r['cfg'].get('axis_block'), threads=THREADS, measured_N=src['cfg']['N'], minutes_per_1e6=src['production_s_median'] * 1e6 / src['cfg']['N'] / 60,
                minutes_per_1e6_scan_only=src['scan_s_median'] * 1e6 / src['cfg']['N'] / 60, minutes_per_1e6_end_to_end=src['end_to_end_s_median'] * 1e6 / src['cfg']['N'] / 60,
                measured_or_extrapolated_1e6=('measured' if src['cfg']['N'] >= 1_000_000 else f"extrapolated_from_{src['cfg']['N']}"), peak_increment_GB=src['peak_increment_GB'], peak_run_increment_GB=src['peak_run_increment_GB'], peak_setup_GB=src['peak_setup_GB'], fallback_tiers=[dict(chunk=a, axis_block=b, measured_N=n) for a, b, n in fb], residency='resident')
cand_l24 = [p for p in (pick('l24_direct_einsum'), pick('l24_feature231')) if p]
def route_tie(cands):    # registered: fastest; within 5% -> lower peak memory -> registered priority (direct before feature)
    if not cands: return None
    fast = min(p['minutes_per_1e6'] for p in cands); band = [p for p in cands if p['minutes_per_1e6'] <= fast / 0.95]; prio = {'l24_direct_einsum': 0, 'l24_feature231': 1}
    return min(band, key=lambda p: (p['peak_increment_GB'], prio[p['route']]))   # raw peak memory, then registered priority
L24 = route_tie(cand_l24)
S2_CPU = pick('s2_feature'); S2_GPU = None; GPU_ADOPT = {}
S2_F32_EQ = bool(XR.get('s2_feature/f32_vs_f64', {}).get('ok', False))
def gpu_eligible(r):
    return (r['status'] == 'ok' and r['hashes_consistent'] and all(np.isfinite(r[f'{k}_s_median']) and r[f'{k}_s_median'] >= 0 for k in ('draw', 'cast', 'h2d', 'scan', 'd2h', 'eval64', 'production', 'end_to_end')) and r['production_s_median'] > 0
            and r['peak_reserved_GB'] <= 0.65 * r['vram_total_GB'] and registered_threads_ok(r))
if FLOAT32_ELIGIBLE and S2_F32_EQ and GPU_STATUS == 'ok':          # evidence chain: CPU S2 f32 ~ CPU S2 f64 AND GPU strict f32 ~ CPU S2 f32
    g = [r for r in gpu_rows if gpu_eligible(r) and not r['cfg']['tf32'] and r['cfg']['stage'] == 'finalist' and XR.get(f"s2_gpu_strict_finalist_ch{r['cfg']['chunk']}_vs_cpu", {}).get('ok')]
    if g:
        r = min(g, key=lambda r: r['production_s_median'] / r['cfg']['N']); o = [x for x in gpu_rows if x['status'] == 'ok' and not x['cfg']['tf32'] and x['cfg']['stage'] == 'optional_1e6' and x['cfg']['chunk'] == r['cfg']['chunk']]
        src = o[0] if o else r; GPU_ADOPT = dict(cross_check_config=f"finalist ch{r['cfg']['chunk']}", inherited_by_optional_1e6=bool(o))
        gpu_min = src['production_s_median'] * 1e6 / src['cfg']['N'] / 60; cpu_min = S2_CPU['minutes_per_1e6'] if S2_CPU else float('inf')
        S2_GPU = dict(route='s2_feature_gpu_strict_fp32_selection_host_float64_eval', selection_dtype='float32', evaluation_dtype='float64', sample_chunk=r['cfg']['chunk'], axis_block=3072, minutes_per_1e6=gpu_min, minutes_per_1e6_scan_only=src['scan_s_median'] * 1e6 / src['cfg']['N'] / 60,
                      speedup_vs_cpu=(cpu_min / gpu_min if gpu_min > 0 else None), accelerated=(gpu_min < cpu_min),
                      measured_N=src['cfg']['N'], peak_alloc_GB=src['peak_alloc_GB'], peak_reserved_over_vram=src['peak_reserved_GB'] / src['vram_total_GB'], gpu=src.get('gpu'), **GPU_ADOPT)
S2_DECISION = dict(baseline_cpu=S2_CPU, optional_gpu=S2_GPU, production_device=('gpu_optional_cpu_baseline' if (S2_GPU and S2_GPU['accelerated']) else 'cpu'), gpu_adoption_rule='A8a both flags and CPU S2 f32-vs-f64 PASS and GPU ok and adopted config cross-check PASS and gpu_eligible (consistent hashes, finite timings, threads registered, peak_reserved <= 65% VRAM); optional 1e6 by linearity(<1.2)+budget')
DEC = dict(l24=L24, s2=S2_DECISION, float32_eligibility=dict(event=F32_EVENT, axis=F32_AXIS, unified_requirement='both'), evaluation_dtype='float64 (frozen, A5 specification)', dtype_rule=f'route-specific: float32 only if A8a float32_eligible_for_rules ({FLOAT32_ELIGIBLE}) and that route f32-vs-f64 cross-check PASS', safe_tie_break='max throughput; within 5% -> smallest (chunk, axis_block)')
GATES['G_route_decision_complete'] = (L24 is not None and S2_CPU is not None)
GATES['G_rules_recommendation_complete'] = GATES['G_route_decision_complete']
REQUIRED = ['G_mt_commit', 'G_mt_origin', 'G_mt_clean', 'G_t2b2_run_sha', 'G_a8a_status', 'G_a8a_manifest_sha', 'G_a8a_manifest_embedded_equal', 'G_F16_file_sha', 'G_a5_bstack_file_sha', 'G_colab_runtime', 'G_F16_array_sha_local',
            'G_registered_inventory_exact', 'G_all_attempts_classified', 'G_all_ok_timings_finite_positive', 'G_memory_telemetry_complete', 'G_thread_configuration_registered', 'G_feasible_config_per_required_route', 'G_chunk_invariance',
            'G_chunk_invariance_coverage', 'G_float64_raw_argmin_exact', 'G_antipode_map_involution', 'G_antipode_map_expected_sha', 'G_eventB_thresholds_bound', 'G_plane_equivalence_gate_selftest', 'G_parent_failed_smoke_binding', 'G_parent_failed_smoke_artifacts', 'G_timed_audit_vectors_saved', 'G_l24_routes_agree_f64', 'G_l24_routes_agree_f32', 'G_gpu_classified', 'G_audit_children_ok', 'G_audit_matches_timed', 'G_route_decision_complete', 'G_rules_recommendation_complete'] + (['G_notebook_live_source'] if A8_MODE == 'official' else ['G_notebook_head_available'])
GATES = {k: bool(v) for k, v in GATES.items()}; EXACT = (set(GATES) == set(REQUIRED)); RUN_PASS = EXACT and all(GATES[k] for k in REQUIRED)
SMOKE_PASS = bool(RUN_PASS and A8_MODE == 'smoke'); BENCHMARK_VALID = bool(RUN_PASS and A8_MODE == 'official'); STATUS = 'SMOKE_PASS' if SMOKE_PASS else 'BENCHMARK_VALID' if BENCHMARK_VALID else 'FAILED'
def row_of(r):
    c = r['cfg']; return dict(stage=c['stage'], route=c['route'], selection_dtype=c['dtype'], N=c['N'], chunk=c['chunk'], axis_block=c.get('axis_block'), status=r['status'], setup_s=r.get('setup_s'), draw_s=r.get('draw_s_median'), cast_s=r.get('cast_s_median'), scan_s=r.get('scan_s_median'), eval64_s=r.get('eval64_s_median'),
                production_s_median=r.get('production_s_median'), production_s_min=r.get('production_s_min'), production_s_max=r.get('production_s_max'), end_to_end_s=r.get('end_to_end_s_median'), per_1e6_min=(r['production_s_median'] * 1e6 / c['N'] / 60 if r['status'] == 'ok' else None), base_rss_GB=r.get('base_rss_GB'), peak_rss_GB=r.get('peak_rss_GB'), peak_increment_GB=r.get('peak_increment_GB'), ru_maxrss_GB=r.get('ru_maxrss_GB'),
                peak_run_increment_GB=r.get('peak_run_increment_GB'), avail_before_setup_GB=r.get('avail_before_setup_GB'), swap_increase_GB=r.get('swap_increase_GB'), eligible=(eligible(r) if r['status'] == 'ok' else False), argmin_hash=(r.get('argmin_hash') or '')[:16], stderr=(r.get('stderr') or '')[:200])
cpu_df = pd.DataFrame([row_of(r) for r in rows]); gpu_df = pd.DataFrame([dict(stage=r['cfg']['stage'], tf32=r['cfg']['tf32'], N=r['cfg']['N'], chunk=r['cfg']['chunk'], status=r['status'], setup_s=r.get('setup_s'), scan_s=r.get('scan_s_median'), eval64_s=r.get('eval64_s_median'), production_s_median=r.get('production_s_median'), end_to_end_s_median=r.get('end_to_end_s_median'),
                                                                               draw_s=r.get('draw_s_median'), cast_s=r.get('cast_s_median'), h2d_s=r.get('h2d_s_median'), d2h_s=r.get('d2h_s_median'), peak_alloc_GB=r.get('peak_alloc_GB'), peak_reserved_GB=r.get('peak_reserved_GB'), gpu=r.get('gpu'), torch=r.get('torch'), cuda=r.get('cuda'), error=r.get('error')) for r in gpu_rows])
for _df, _nm in ((cpu_df, 'a8b_benchmark_cpu.csv'), (gpu_df, 'a8b_benchmark_gpu.csv')):
    _t = os.path.join(OUT, _nm + '.tmp'); _df.to_csv(_t, index=False); os.replace(_t, os.path.join(OUT, _nm))
with open(os.path.join(OUT, 'a8b_attempts_evidence.jsonl.tmp'), 'w') as fh:
    for r in rows + gpu_rows:
        ev = {k: v for k, v in r.items() if k not in ('audit_sel', 'audit_T1', 'audit_T2', 'audit_argmin', 'audit_margin')}
        if 'audit_sel' in r: ev['audit_summary'] = dict(n=len(r['audit_sel']), sel_sha256=hashlib.sha256(np.array(r['audit_sel']).tobytes()).hexdigest(), T1_sha256=hashlib.sha256(np.array(r['audit_T1']).tobytes()).hexdigest(), T2_sha256=hashlib.sha256(np.array(r['audit_T2']).tobytes()).hexdigest(), margin_min=(float(min(r['audit_margin'])) if 'audit_margin' in r else None), margin_median=(float(np.median(r['audit_margin'])) if 'audit_margin' in r else None))
        fh.write(json.dumps(ev) + chr(10))
os.replace(os.path.join(OUT, 'a8b_attempts_evidence.jsonl.tmp'), os.path.join(OUT, 'a8b_attempts_evidence.jsonl'))
prov = dict(notebook=f'Step1 Phase A-8b scan benchmark v1.1.8 [{A8_MODE}]', mode=A8_MODE, status=STATUS, RUN_PASS=RUN_PASS, SMOKE_PASS=SMOKE_PASS, BENCHMARK_VALID=BENCHMARK_VALID, timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
            gates=GATES, required_gates=REQUIRED, gate_inventory_exact=EXACT, notebook_identity=DIAG['notebook'], environment=ENV, child_script_sha256=CHILD_SHA, gpu_child_script_sha256=GPU_CHILD_SHA,
            inputs=dict(a8a_provenance_sha256=A8A_PROV_SHA, a8a_status=a8a['status'], manifest_sha256=a8a['manifest_file_sha256'], expected=EXPECTED, float32_eligible_for_event_outputs=F32_EVENT, float32_eligible_for_axis_outputs=F32_AXIS, float32_selection_allowed=FLOAT32_ELIGIBLE, mt_commit=MT_COMMIT),
            tolerances_eval=TOL_EVAL, registered=dict(design='staged: pilot(all, N=%s, rep1) -> finalist(top-%s, N=%d, rep3) -> winner(top-1, N=%s, rep3) -> optional S2 1e6 by linearity(<1.2)+budget(%ds)' % (N_PILOT, TOPK, N_FINAL, N_WIN, BUDGET_S2_1E6_S),
                            pilot=REGISTERED_PILOT, finalist=REGISTERED_FINAL, winner=REGISTERED_WIN, optional=OPT, audit_n=AUDIT_N, seed=SEED, threads=THREADS, warmup='2 chunks',
                            eligibility='ok & consistent hashes & peak_increment <= 0.65 x available_before_setup & swap_increase == 0', tie_break='max throughput; ties within 5% -> (chunk, axis_block) ascending', tolerances=TOL),
            chunk_invariance=inv, cross_route=XR, gpu_status=GPU_STATUS, ROUTE_DECISION=DEC, diagnostics=DIAG,
            selection_reproducibility=dict(measured=dict(float64_raw_argmin_exact=GATES['G_float64_raw_argmin_exact'],
                                                         float32_flip_occurrences_across_chunk_comparisons=int(sum(round(x['flip_frac'] * len(AUDITS[(k.split('/')[0], 'float32')]['audit_argmin'])) for k, v in inv.items() if k.endswith('/float32') for x in v['detail'])),   # config-wise occurrences: the same sample is counted once per timed config
                                                         float32_unique_flipped_samples_by_route={k: int(len(set().union(*[set(np.flatnonzero(np.array(r['audit_argmin']) != np.array(AUDITS[(k.split('/')[0], 'float32')]['audit_argmin'])).tolist()) for r in okr if (r['cfg']['route'], r['cfg']['dtype']) == (k.split('/')[0], 'float32') and 'audit_argmin_sha' in r] or [set()]))) for k in inv if k.endswith('/float32')},
                                                         float32_all_flips_antipodal=all(v['all_flips_antipodal'] for k, v in inv.items() if k.endswith('/float32')),
                                                         float32_max_selection_score_rel=max([x['sel_rel'] for k, v in inv.items() if k.endswith('/float32') for x in v['detail']] or [0.0]),
                                                         float32_max_T1_rel=max([x['t1_rel'] for k, v in inv.items() if k.endswith('/float32') for x in v['detail']] or [0.0]),
                                                         float32_max_T2_rel=max([x['t2_rel'] for k, v in inv.items() if k.endswith('/float32') for x in v['detail']] or [0.0]),
                                                         eventB_identical_all=all(x['eventB_identical'] for v in inv.values() for x in v['detail']), float32_selection_present=('float32' in DTYPES),
                                                         eventB_indicator_nondegenerate_in_audit={f'{k[0]}/{k[1]}': bool(len(set(((np.array(v['audit_T1']) <= T1o) & (np.array(v['audit_T2']) <= T2o)).tolist())) == 2) for k, v in AUDITS.items()}),
                                           eventB_note='A8b benchmark audit samples are unit-variance synthetic draws (not physical muK^2 amplitude), so the Event B indicator is degenerate (all True) here. A8b checks the helper, the threshold binding and per-sample indicator equality only; '
                                                       'A8b does NOT demonstrate physical Event B invariance. Float32 Event B eligibility on physical-amplitude samples is inherited from A8a official (G_F16_f32_eventB / float32_eligible_for_event_outputs).',
                                           margin_definition='raw_oriented_axis_margin: (second-best - best)/|best| over the 3072 oriented axes excluding only the winner (the antipodal duplicate is NOT excluded; equals 0 at exact antipodal ties)',
                                           antipode_map_sha256=DIAG['antipode_map_sha256'], gate_selftest=DIAG['gate_selftest'], eventB_thresholds=DIAG['eventB_thresholds']),
            amendment=AMEND,
            RULES_RECOMMENDATION=dict(note='engineering input for rules v1.0; S2 scientific adoption still requires the S4/exact validation gate; GPU is an optional accelerated route, CPU is the required baseline', **DEC,
                                      axis_reproducibility=dict(scientific_axis='unoriented mirror plane [n] = {n, -n}', raw_pixel_index='not portable across float32 blocking / BLAS / hardware at exact antipodal ties',
                                                                selection_configuration=dict(l24=(dict(route=L24['route'], selection_dtype=L24['selection_dtype'], sample_chunk=L24['sample_chunk'], axis_block=L24['axis_block'], blas_threads=THREADS) if L24 else None),
                                                                                             s2=(dict(route=S2_CPU['route'], selection_dtype=S2_CPU['selection_dtype'], sample_chunk=S2_CPU['sample_chunk'], axis_block=S2_CPU['axis_block'], blas_threads=THREADS) if S2_CPU else None)),
                                                                evaluation_dtype='float64', float64_selection='raw axis index reproduction required across registered chunks',
                                                                statement='S2 axis selection uses the registered route, selection dtype, sample chunk, axis block and BLAS thread count. In float32, when an antipodal axis pair is an exact implemented tie '
                                                                          '(identical reflection rows), the raw HEALPix pixel index may swap between blocking configurations; the scientific axis output is the unoriented mirror-plane class [n]={n,-n}. '
                                                                          'T1, T2 and Event B are evaluated on the selected plane with the frozen float64 B-stack. Freezing chunk/threads does NOT make the raw index portable across CPU/BLAS/hardware.')),
            run_id=RUN_ID, bench_env=BENCH_ENV, audits={f'{k[0]}/{k[1]}': dict(argmin_sha=v['audit_argmin_sha'], raw_oriented_axis_margin_min=float(min(v['audit_margin'])), raw_oriented_axis_margin_median=float(np.median(v['audit_margin'])), cfg={kk: vv for kk, vv in v['cfg'].items() if kk in ('chunk', 'axis_block')}) for k, v in AUDITS.items()}, outputs=dict(cpu_csv_sha256=sha256_file(os.path.join(OUT, 'a8b_benchmark_cpu.csv')), gpu_csv_sha256=sha256_file(os.path.join(OUT, 'a8b_benchmark_gpu.csv')), evidence_jsonl_sha256=sha256_file(os.path.join(OUT, 'a8b_attempts_evidence.jsonl')), audit_vectors_npz_sha256=sha256_file(os.path.join(OUT, 'a8b_audit_vectors.npz')), timed_audit_vectors_npz_sha256=sha256_file(os.path.join(OUT, 'a8b_timed_audit_vectors.npz')), timed_audit_index_json_sha256=sha256_file(os.path.join(OUT, 'a8b_timed_audit_index.json'))), checkpoint_binding=BINDING, git_calls=GIT_LOG)
atomic_json(prov, os.path.join(OUT, 'a8b_provenance.json'))
print(cpu_df[['stage', 'route', 'selection_dtype', 'N', 'chunk', 'axis_block', 'status', 'scan_s', 'eval64_s', 'production_s_median', 'per_1e6_min', 'peak_increment_GB', 'eligible']].to_string(index=False, float_format=lambda v: f'{v:.3g}'))
print('\nROUTE_DECISION:', json.dumps(DEC, indent=1)[:1800]); print('STATUS =', STATUS)
failed = {k: v for k, v in GATES.items() if not v}
assert RUN_PASS, failed
if A8_MODE == 'smoke': assert SMOKE_PASS and not BENCHMARK_VALID, STATUS
else: assert BENCHMARK_VALID and not SMOKE_PASS, STATUS

## 実行手順（v1.1.8・amended analysis）
0. Drive の `runs_step1_phaseA/` に **`A8b_amendment_v1.1.7.md`** を置く（provenance が SHA を記録）。`a8b_v1.1.5_smoke/`（FAILED）は**そのまま残す**（上書き・削除しない）。
1. 本ノートブック（v1.1.8）を commit・push（`A8b_amendment_v1.1.7.md` も同じ commit に追加）。2. **fresh runtime**（可能なら GPU）で smoke（先頭に `A8_MODE='smoke'` セルを追加・数分）→ `SMOKE_PASS`。
3. 再び fresh runtime で純正 v1.1.8 を Runtime restart → Run all（official・A8a v1.1.2 official の artifact を使用）。
返送：`a8b_v1.1.8_smoke/` と `a8b_v1.1.8_official/` の `a8b_benchmark_cpu.csv`・`a8b_benchmark_gpu.csv`・`a8b_provenance.json`・`a8b_audit_vectors.npz`・**`a8b_timed_audit_vectors.npz`・`a8b_timed_audit_index.json`**・`a8b_attempts_evidence.jsonl`・全セル出力。
